<a href="https://colab.research.google.com/github/lander302/HonoursProject_new/blob/main/HonoursTimeSeriesCode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#Import libraries
import pandas as pd
import numpy as np

# Load existing Power BI dataset
df = pd.read_csv("students_with_predictions_old.csv")

# Generate random numbers
rng = np.random.default_rng(42)

# Create 12 weekly dates (Mondays)
weeks = pd.date_range("2025-01-06", periods=12, freq="W-MON")

rows = []
for _, r in df.iterrows():
    base_att = float(r["Attendance (%)"])
    base_eng = float(r["Participation_Score"])
    base_stress = float(r["Stress_Level (1-10)"])

    for w in weeks:
        # Weekly attendance fluctuates around baseline
        att_w = np.clip(base_att + rng.normal(0, 3), 40, 100)

        # Engagement fluctuates; higher stress slightly reduces it
        eng_w = np.clip(base_eng + rng.normal(0, 6) - (base_stress - 5) * 1.2, 0, 100)

        # Behaviour incidents per week: more likely if stress high + attendance low
        lam = max(0.2, 0.6 + 0.12*(base_stress - 5) + 0.01*(80 - att_w))
        beh_w = rng.poisson(lam)

        rows.append({
          "Student_ID": r["Student_ID"],
          "Week_Start": w,
          "Attendance_Weekly": round(att_w, 2),
          "Engagement_Weekly": round(eng_w, 2),
          "Behaviour_Incidents_Weekly": int(beh_w),
        })

student_weekly = pd.DataFrame(rows)

# A MonthStart column was added for monthly visuals in Power BI for teachers to filter between
student_weekly["MonthStart"] = student_weekly["Week_Start"].dt.to_period("M").dt.to_timestamp()


# Save as a new timestamp file to add to powerbi
student_weekly.to_csv("student_weekly_metrics.csv", index=False)
print("Saved: student_weekly_metrics.csv", student_weekly.shape)
print(student_weekly.head())


Saved: student_weekly_metrics.csv (72000, 6)
  Student_ID Week_Start  Attendance_Weekly  Engagement_Weekly  \
0      S1000 2025-01-06              79.83              54.54   
1      S1000 2025-01-13              75.01              61.55   
2      S1000 2025-01-20              76.36              66.06   
3      S1000 2025-01-27              80.32              55.62   
4      S1000 2025-02-03              81.56              60.48   

   Behaviour_Incidents_Weekly MonthStart  
0                           2 2025-01-01  
1                           1 2025-01-01  
2                           2 2025-01-01  
3                           1 2025-01-01  
4                           1 2025-02-01  
